<a href="https://colab.research.google.com/github/barr92/ab_testing_project/blob/main/ireland_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ireland property scraper — sales and rentals

One Python 3.10+ notebook; no third-party packages required by the scraper. Collects advertised asking prices and rents, **not completed sale transactions**. Local SQLite checkpoints, deduplicated CSV/JSONL exports and a separate search-area membership export.

**Site inspection: 12 November 2026.** Public raw HTML validated for Dublin sales page 1, sales page 2, rentals page 1, and one brochure in each channel. Search data is server-rendered in `script#ng-state`, under `SEARCH_RESOLVER:<path>`, with `SearchResults`, `Page`, `PageSize`, `ResultCount`. Brochures use `BROCHURE_RESOLVER:<path> → Brochure → Property`. No private API, embedded API credentials, guessed CSS classes or browser circumvention is used.

**Terms:** [current terms](https://news.myhome.ie/uncategorized/terms-and-conditions-36589), section 7, restrict collecting property details and copying content without prior written consent. Obtain consent covering your collection and reuse before enabling network collection. [robots.txt](https://www.myhome.ie/robots.txt) currently permits bare `?page=N` pagination but restricts other listing query strings; robots permission does not grant reuse rights. Review both again before use. The notebook does not establish legal permission.

**Validation boundary:** parser validation used downloaded public HTML, including actual nonzero map coordinates in sales and rental search records. Full national collection, Smartproxy credentials/connectivity, long-running recovery and every county/city endpoint were not live-tested. Missing map coordinates stay null. Some brochure pages omit coordinates; unrelated price-register, agent and nearby-property coordinates are deliberately ignored.

The attached reference notebook was recovered from the linked chat and inspected. Its inspect-first, per-area persistence and resume ideas are retained. UK-specific location identifiers, date cutoffs, enrichment, cloud project names and database promotion are not carried over.


## 1. Configure

Run all cells with `RUN_NETWORK=False` first. Configure environment variables **outside the notebook**, then restart the kernel so they are available. Never paste secrets into cells or saved outputs.

Required for proxy mode: `SMARTPROXY_USERNAME`, `SMARTPROXY_PASSWORD`, `SMARTPROXY_HOST`, `SMARTPROXY_PORT`. Use the host and port supplied in your provider dashboard; no endpoint is assumed. `SMARTPROXY_SCHEME` defaults to `http` (HTTP CONNECT for HTTPS targets). URL-encoded credentials are handled automatically. Set `USE_PROXY=False` for a permitted direct connection. Proxy errors are redacted; no rotation or retry on access denials.

After reviewing terms and obtaining consent, set `RUN_NETWORK=True`, keep `INSPECT_ONLY=True`, and inspect one page per channel. For the full run set `INSPECT_ONLY=False` and use a **different** `RUN_ID`. Keep the same run ID to resume an interrupted run; use a fresh ID for a new snapshot. Persist `OUTPUT_DIR` on durable storage if running in Colab.

The final section now uploads the saved checkpoint into BigQuery. Colab Secrets are loaded automatically before collection.


In [ ]:
import os, json, re, csv, math, time, random, sqlite3, hashlib, ssl
from pathlib import Path
from datetime import datetime, timezone
from html.parser import HTMLParser
from urllib.parse import urljoin, urlsplit, urlunsplit, quote, parse_qs
from urllib.request import Request, build_opener, ProxyHandler, HTTPSHandler, HTTPRedirectHandler
from urllib.error import HTTPError, URLError

RUN_NETWORK = False
INSPECT_ONLY = True
USE_PROXY = True
RUN_ID = 'inspection-2026-11-12'  # change for every new snapshot
OUTPUT_DIR = Path('ireland_output')
MAX_PAGES_PER_AREA = 1000
MIN_REQUEST_INTERVAL = 3.0
JITTER_SECONDS = 1.0
MAX_ATTEMPTS = 3
TIMEOUT_SECONDS = 45
FETCH_DETAILS = True
# None = every discovered county/city. Example: {'dublin', 'cork-city'}
AREA_SLUGS = None
BASE = 'https://www.myhome.ie'
CHANNELS = {'sale': ('residential', 'property-for-sale'),
            'rent': ('rentals', 'property-to-rent')}
COUNTIES = set(('Antrim Armagh Carlow Cavan Clare Cork Derry Donegal Down Dublin '
    'Fermanagh Galway Kerry Kildare Kilkenny Laois Leitrim Limerick Longford '
    'Louth Mayo Meath Monaghan Offaly Roscommon Sligo Tipperary Tyrone '
    'Waterford Westmeath Wexford Wicklow').lower().split())
def utcnow(): return datetime.now(timezone.utc).isoformat()
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID), 'Use a simple run ID'
assert MAX_PAGES_PER_AREA > 0 and MIN_REQUEST_INTERVAL >= 1
print('Configuration ready. Network enabled:', RUN_NETWORK)


Configuration ready. Network enabled: False


### Colab Secrets
When network collection and proxy mode are enabled, this reads the four existing Colab Secrets.
Enable Notebook access for each. Existing environment settings remain supported outside Colab.


In [ ]:
if RUN_NETWORK and USE_PROXY:
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    required = ['SMARTPROXY_USERNAME','SMARTPROXY_PASSWORD','SMARTPROXY_HOST','SMARTPROXY_PORT']
    for name in required:
        if userdata is not None:
            try:
                value = userdata.get(name)
                if value: os.environ[name] = value.strip()
            except Exception:
                pass
    missing = [name for name in required if not os.environ.get(name)]
    if missing:
        raise RuntimeError('Missing proxy settings: '+', '.join(missing)+
                           '. Check Colab Secrets names and Notebook access.')
    print('All four proxy settings loaded successfully.')


## 2. Public HTML adapters

Only the observed resolver payloads are parsed. A missing or changed payload raises a schema error; it is never treated as a successful empty search. Map coordinates are taken only from the current listing's `BrochureMap` or its own JSON-LD `geo`, with provenance and the site's accuracy flag retained. `(0, 0)` and values outside Ireland's broad bounding box are rejected. No geocoding, city-centroid substitution or inference from an address.


In [ ]:
class SchemaChanged(RuntimeError): pass
class StopCollection(RuntimeError): pass

class PublicHTML(HTMLParser):
    def __init__(self, html):
        super().__init__(); self.scripts=[]; self.links=[]; self.active=None; self.parts=[]
        self.feed(html)
    def handle_starttag(self, tag, attrs):
        a=dict(attrs)
        if tag == 'script': self.active=a; self.parts=[]
        if tag == 'a' and a.get('href'): self.links.append(a['href'])
    def handle_data(self, data):
        if self.active is not None: self.parts.append(data)
    def handle_endtag(self, tag):
        if tag == 'script' and self.active is not None:
            self.scripts.append((self.active, ''.join(self.parts))); self.active=None

def resolver(html, prefix):
    doc=PublicHTML(html)
    scripts=[s for a,s in doc.scripts if a.get('id') == 'ng-state']
    if len(scripts) != 1: raise SchemaChanged('Expected exactly one ng-state payload')
    try: state=json.loads(scripts[0])
    except ValueError: raise SchemaChanged('Invalid ng-state JSON') from None
    matches=[v for k,v in state.items() if k.startswith(prefix+':')]
    if len(matches) != 1 or not isinstance(matches[0],dict):
        raise SchemaChanged('Expected one '+prefix+' payload')
    return matches[0], doc

def parse_search(html, expected_page):
    data, doc=resolver(html, 'SEARCH_RESOLVER')
    for k in ('SearchResults','Page','PageSize','ResultCount'):
        if k not in data: raise SchemaChanged('Missing search field: '+k)
    if not isinstance(data['SearchResults'],list): raise SchemaChanged('SearchResults is not a list')
    if data['Page'] != expected_page: raise SchemaChanged('Server returned a different page')
    if not isinstance(data['PageSize'],int) or data['PageSize'] <= 0:
        raise SchemaChanged('Invalid PageSize')
    if not isinstance(data['ResultCount'],int) or data['ResultCount'] < 0:
        raise SchemaChanged('Invalid ResultCount')
    if data['ResultCount'] > 0 and not data['SearchResults']:
        raise SchemaChanged('Empty results despite positive advertised total')
    for p in data['SearchResults']:
        if not isinstance(p,dict) or not p.get('PropertyId') or not p.get('BrochureUrl'):
            raise SchemaChanged('Missing listing identity or brochure URL')
    return data

def valid_geo(lat, lon):
    try:
        lat,lon=float(lat),float(lon)
        return (lat,lon) if 51 <= lat <= 56 and -11 <= lon <= -5 else (None,None)
    except (TypeError,ValueError): return None,None

def number_text(text):
    m=re.search(r'\d+',str(text or ''))
    return int(m[0]) if m else None

def asking_price(text):
    # Preserve ranges/POA separately; never mistake a range for an exact price.
    text=str(text or '')
    m=re.fullmatch(r'\s*€\s*([\d,]+(?:\.\d+)?)\s*(?:/\s*(month|week|year|day))?\s*',text,re.I)
    return (float(m[1].replace(',','')),m[2].lower() if m[2] else None) if m else (None,None)

def listing_record(p, channel):
    amount,period=asking_price(p.get('PriceAsString'))
    geo=p.get('BrochureMap') or {}
    lat,lon=valid_geo(geo.get('latitude'),geo.get('longitude'))
    href=p['BrochureUrl']; url=urljoin(BASE,href)
    if urlsplit(url).netloc != urlsplit(BASE).netloc or not re.fullmatch(
        r'/(residential|rentals)/brochure/[^/?#]+/\d+',urlsplit(url).path):
        raise SchemaChanged('Unexpected brochure URL')
    return dict(listing_id=str(p['PropertyId']),channel=channel,source_url=url,
        address=p.get('DisplayAddress'),eircode=None,locality=None,region=None,
        price_text=p.get('PriceAsString'),asking_price_eur=amount,rent_period=period,
        bedrooms=p.get('NumberOfBeds'),bathrooms=p.get('NumberOfBathrooms'),
        floor_area_m2=p.get('SizeStringMeters'),floor_area_ft2=p.get('SizeStringFeet'),
        property_type=p.get('PropertyType'),ber_rating=p.get('BerRating'),
        listing_status=p.get('PropertyStatus'),is_sale_agreed=p.get('IsSaleAgreed'),
        is_private_landlord=p.get('IsPrivateLandlord'),
        agent_name=None if p.get('IsPrivateLandlord') else p.get('GroupName'),
        created_on=p.get('CreatedOnDate'),activated_on=p.get('ActivatedOn'),
        refreshed_on=p.get('RefreshedOn'),latitude=lat,longitude=lon,
        coordinate_source='search.BrochureMap' if lat is not None else None,
        is_mapped_accurately=p.get('IsMappedAccurately'),detail_status='not_requested',
        collected_at=utcnow())

def enrich_detail(row, html):
    data,doc=resolver(html,'BROCHURE_RESOLVER')
    p=(data.get('Brochure') or {}).get('Property')
    if not isinstance(p,dict) or str(p.get('PropertyId')) != row['listing_id']:
        raise SchemaChanged('Brochure identity mismatch')
    for key,field in [('eircode','Eircode'),('locality','LocalityName'),('region','RegionName')]:
        row[key]=p.get(field)
    for key,field in [('bedrooms','BedsString'),('bathrooms','BathString')]:
        if row[key] is None: row[key]=number_text(p.get(field))
    # Only a JSON-LD entity explicitly identified as this property can provide geo.
    for attrs,raw in doc.scripts:
        if attrs.get('type') != 'application/ld+json': continue
        try: obj=json.loads(raw)
        except ValueError: continue
        if not isinstance(obj,dict): continue
        if obj.get('@id') != row['source_url']+'#property': continue
        geo=obj.get('geo') or {}
        lat,lon=valid_geo(geo.get('latitude'),geo.get('longitude'))
        if row['latitude'] is None and lat is not None:
            row.update(latitude=lat,longitude=lon,coordinate_source='detail.JSON-LD.geo')
    row['detail_status']='ok'
    return row


## 3. Discover counties and cities from actual links

The crawler reads the homepage's existing search links rather than constructing unverified county or city slugs. It selects traditional county names, explicitly named `*-city` areas and Dublin city-centre areas if linked. Clare and Roscommon use observed `clare-1` and `roscommon-1` slugs. Cork City uses an observed `-in-cork-city` route. The homepage's all-property Louth link incorrectly points to Mayo: two independently verified Louth search URLs are supplied as documented corrections. All other discovered areas are saved in the catalogue and may be selected explicitly with `AREA_SLUGS`.

Coverage is the site's search geography, **not administrative boundaries**. Dublin county is not labelled Dublin city. Counties without site links (including any Northern Ireland counties not offered) are reported as unavailable. Additional cities such as Waterford/Kilkenny are not assumed to have a separate city endpoint. Review the catalogue and missing-area report; a national total cannot be promised. County/city overlap is preserved in memberships and deduplicated in listing exports.


In [ ]:
def discover_areas(html):
    found={}
    for href in PublicHTML(html).links:
        u=urlsplit(urljoin(BASE,href))
        if u.netloc != urlsplit(BASE).netloc or u.query: continue
        for channel,(section,tail) in CHANNELS.items():
            m=re.fullmatch('/'+section+r'/([^/]+)/'+tail,u.path)
            if m:
                slug=m[1]
                if slug == 'ireland': continue
                county_slug={'clare-1':'clare','roscommon-1':'roscommon'}.get(slug,slug)
                kind='county' if county_slug in COUNTIES else ('city_search' if
                    slug.endswith('-city') or slug in {'north-city-centre','south-city-centre'} else 'other')
                found[(channel,slug)]=dict(channel=channel,slug=slug,county_slug=county_slug,kind=kind,
                    name=slug.replace('-',' ').title(),url=urlunsplit((u.scheme,u.netloc,u.path,'','')))
            city=re.fullmatch('/'+section+r'/([^/]+)/'+tail+r'-in-([^/]+-city)',u.path)
            if city:
                slug=city[2]
                found[(channel,slug)]=dict(channel=channel,slug=slug,county_slug=None,
                    kind='city_search',name=slug.replace('-',' ').title(),url=urlunsplit((u.scheme,u.netloc,u.path,'','')))
    if not found: raise SchemaChanged('No observed search links on homepage')
    # Homepage currently mislinks Louth's all-property anchor to Mayo. Both corrected
    # all-property endpoints below were independently opened and verified 2026-11-12.
    for channel,(section,tail) in CHANNELS.items():
        found[(channel,'louth')]=dict(channel=channel,slug='louth',county_slug='louth',
            kind='county',name='Louth',url=BASE+'/'+section+'/louth/'+tail)
    return sorted(found.values(),key=lambda a:(a['channel'],a['slug']))


## 4. Rate-limited transport

Sequential requests, minimum interval plus jitter, timeout and bounded backoff for 5xx/network errors. HTTP 401/403/407/429 immediately stop the run; do not switch proxies to bypass a denial. Redirects are restricted to this host and are also rate limited. The notebook uses only canonical search URLs and bare page parameters, matching the robots rules inspected above. It does not fetch private/account paths, arbitrary queries or internal endpoints. Check for changed robots rules before a new run.


In [ ]:
class Transport:
    def __init__(self):
        self.last=0.0
        proxies={}
        if USE_PROXY:
            names=['SMARTPROXY_USERNAME','SMARTPROXY_PASSWORD','SMARTPROXY_HOST','SMARTPROXY_PORT']
            if any(not os.environ.get(n) for n in names):
                raise StopCollection('Missing proxy configuration environment variables')
            scheme=os.environ.get('SMARTPROXY_SCHEME','http')
            host=os.environ['SMARTPROXY_HOST']; port=os.environ['SMARTPROXY_PORT']
            if scheme not in {'http','https'} or not re.fullmatch(r'[A-Za-z0-9.-]+',host) or not port.isdigit():
                raise StopCollection('Invalid proxy host, port or scheme')
            proxy=(scheme+'://'+quote(os.environ[names[0]],safe='')+':'+
                quote(os.environ[names[1]],safe='')+'@'+host+':'+port)
            proxies={'http':proxy,'https':proxy}
        transport=self
        class SafeRedirect(HTTPRedirectHandler):
            def redirect_request(self, req, fp, code, msg, headers, newurl):
                if urlsplit(newurl).netloc != urlsplit(BASE).netloc:
                    raise StopCollection('Cross-host redirect refused')
                transport.wait()
                return super().redirect_request(req,fp,code,msg,headers,newurl)
        self.opener=build_opener(ProxyHandler(proxies),HTTPSHandler(context=ssl.create_default_context()),SafeRedirect())
    def wait(self):
        pause=MIN_REQUEST_INTERVAL+random.uniform(0,JITTER_SECONDS)-(time.monotonic()-self.last)
        if pause>0: time.sleep(pause)
        self.last=time.monotonic()
    def get(self,url):
        if urlsplit(url).netloc != urlsplit(BASE).netloc: raise StopCollection('Unexpected host')
        for attempt in range(MAX_ATTEMPTS):
            self.wait()
            try:
                req=Request(url,headers={'User-Agent':'IrelandPropertyResearch/1.0','Accept':'text/html'})
                with self.opener.open(req,timeout=TIMEOUT_SECONDS) as response:
                    return response.read().decode('utf-8-sig')
            except HTTPError as e:
                status=e.code
                if status in {401,403,407,429}:
                    raise StopCollection('Access/rate limit response: HTTP '+str(status)) from None
                if status<500 or attempt+1==MAX_ATTEMPTS:
                    raise StopCollection('Request failed: HTTP '+str(status)) from None
            except (URLError,TimeoutError,OSError):
                if attempt+1==MAX_ATTEMPTS:
                    raise StopCollection('Network/proxy request failed; details redacted') from None
            time.sleep(min(60,2**(attempt+2)))


## 5. Transactional checkpoints and exports

Each page's listings, area memberships and next-page checkpoint commit in one SQLite transaction. An interruption before commit replays that page; primary keys prevent duplicates. Failed pages do not advance. Repeated page signatures stop with an explicit incomplete status. Reaching the configured cap is marked `capped`, never complete. On rerun, capped searches continue from their next page if you increase the cap. Exports are rebuilt from SQLite and replaced atomically.

SQLite is local to this run ID. One row per `(channel, listing_id)` per run; separate run files retain observations over time. Search membership is not inferred property geography. Keep SQLite files for resumability. Do not run concurrent kernels against the same run ID.


In [ ]:
def open_store(path):
    db=sqlite3.connect(path)
    db.executescript("""
    CREATE TABLE IF NOT EXISTS listings(channel TEXT,id TEXT,payload TEXT,PRIMARY KEY(channel,id));
    CREATE TABLE IF NOT EXISTS memberships(channel TEXT,slug TEXT,id TEXT,PRIMARY KEY(channel,slug,id));
    CREATE TABLE IF NOT EXISTS progress(channel TEXT,slug TEXT,next_page INTEGER,status TEXT,total INTEGER,
      PRIMARY KEY(channel,slug));
    CREATE TABLE IF NOT EXISTS pages(channel TEXT,slug TEXT,page INTEGER,signature TEXT,
      PRIMARY KEY(channel,slug,page));
    CREATE TABLE IF NOT EXISTS metadata(key TEXT PRIMARY KEY,value TEXT);
    """)
    return db

def atomic_text(path,text):
    temp=path.with_suffix(path.suffix+'.tmp'); temp.write_text(text,encoding='utf-8'); temp.replace(path)

def export_store(db, folder, run_id):
    rows=[json.loads(r[0]) for r in db.execute('SELECT payload FROM listings ORDER BY channel,id')]
    for channel in ['all','sale','rent']:
        selected=[r for r in rows if channel=='all' or r['channel']==channel]
        prefix=folder/('ireland_'+run_id+'_'+channel)
        atomic_text(prefix.with_suffix('.jsonl'),''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in selected))
        columns=list(rows[0]) if rows else ['listing_id','channel','latitude','longitude']
        tmp=prefix.with_suffix('.csv.tmp')
        with tmp.open('w',newline='',encoding='utf-8-sig') as f:
            w=csv.DictWriter(f,fieldnames=columns); w.writeheader(); w.writerows(selected)
        tmp.replace(prefix.with_suffix('.csv'))
    memberships=[dict(channel=c,search_area=s,listing_id=i) for c,s,i in
        db.execute('SELECT channel,slug,id FROM memberships ORDER BY channel,slug,id')]
    atomic_text(folder/('ireland_'+run_id+'_memberships.jsonl'),
        ''.join(json.dumps(r)+'\n' for r in memberships))
    progress=[dict(channel=c,search_area=s,next_page=p,status=t,advertised_total=n) for c,s,p,t,n in
        db.execute('SELECT * FROM progress ORDER BY channel,slug')]
    atomic_text(folder/('ireland_'+run_id+'_coverage.json'),json.dumps(progress,indent=2))
    with_geo=sum(r['latitude'] is not None for r in rows)
    print(f'Exported {len(rows)} unique listings; {with_geo} with coordinates; {len(rows)-with_geo} missing.')
    print('Search status:',progress[:10], '(first 10)')

def crawl_area(db, client, area, page_cap, details):
    channel,slug=area['channel'],area['slug']
    checkpoint=db.execute('SELECT next_page,status FROM progress WHERE channel=? AND slug=?',(channel,slug)).fetchone()
    if checkpoint and checkpoint[1]=='complete': return
    page=checkpoint[0] if checkpoint else 1
    while page<=page_cap:
        url=area['url']+('' if page==1 else '?page='+str(page))
        data=parse_search(client.get(url),page)
        props=data['SearchResults']; total=data['ResultCount']
        signature=hashlib.sha256(','.join(sorted(str(p['PropertyId']) for p in props)).encode()).hexdigest()
        repeated=db.execute('SELECT 1 FROM pages WHERE channel=? AND slug=? AND signature=?',
            (channel,slug,signature)).fetchone()
        if repeated:
            with db:
                db.execute('INSERT OR REPLACE INTO progress VALUES(?,?,?,?,?)',(channel,slug,page,'repeated_page',total))
            raise SchemaChanged('Repeated page for '+channel+'/'+slug+'; coverage incomplete')
        rows=[]
        for p in props:
            old=db.execute('SELECT payload FROM listings WHERE channel=? AND id=?',(channel,str(p['PropertyId']))).fetchone()
            # Avoid refetching details for duplicate county/city appearances within the same snapshot.
            row=json.loads(old[0]) if old else listing_record(p,channel)
            if not old and details: row=enrich_detail(row,client.get(row['source_url']))
            rows.append(row)
        last=page>=math.ceil(total/data['PageSize'])
        status='complete' if last else ('capped' if page==page_cap else 'running')
        with db:
            for row in rows:
                db.execute('INSERT OR IGNORE INTO listings VALUES(?,?,?)',(channel,row['listing_id'],json.dumps(row)))
                db.execute('INSERT OR IGNORE INTO memberships VALUES(?,?,?)',(channel,slug,row['listing_id']))
            db.execute('INSERT OR REPLACE INTO pages VALUES(?,?,?,?)',(channel,slug,page,signature))
            db.execute('INSERT OR REPLACE INTO progress VALUES(?,?,?,?,?)',(channel,slug,page+1,status,total))
        print(channel,slug,'page',page,'rows',len(rows),'status',status)
        if last: return
        page+=1


## 6. Run inspection or collection

With network collection disabled, this cell performs no requests and writes no files. Inspection selects one Dublin page per channel (or the first selected area available), includes optional brochure enrichment, and exports to its own run ID. Inspect missing coordinates, price periods and coverage before a full run. Search results can change while paging; advertised totals and actual counts need not match. This notebook does not declare missing listings sold or withdrawn.


In [ ]:
def run_collection():
    client=Transport()
    catalogue=discover_areas(client.get(BASE+'/'))
    areas=[a for a in catalogue if (a['slug'] in AREA_SLUGS if AREA_SLUGS is not None
                                   else a['kind'] in {'county','city_search'})]
    if not areas: raise StopCollection('No selected areas are linked by the site')
    unavailable={c:sorted(COUNTIES-{a['county_slug'] for a in catalogue if a['channel']==c and a['kind']=='county'}) for c in CHANNELS}
    print('Counties without discovered endpoints:',unavailable)
    if INSPECT_ONLY:
        areas=[next((a for a in areas if a['channel']==c and a['slug']=='dublin'),
                    next((a for a in areas if a['channel']==c),None)) for c in CHANNELS]
        areas=[a for a in areas if a]
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    atomic_text(OUTPUT_DIR/('ireland_'+RUN_ID+'_areas.json'),json.dumps(
        dict(catalogue=catalogue,selected=areas,missing_counties=unavailable),indent=2))
    db=open_store(OUTPUT_DIR/('ireland_'+RUN_ID+'_checkpoint.sqlite'))
    settings=json.dumps(dict(inspect=INSPECT_ONLY,details=FETCH_DETAILS,areas=areas),sort_keys=True)
    previous=db.execute("SELECT value FROM metadata WHERE key='settings'").fetchone()
    if previous and previous[0]!=settings:
        db.close(); raise StopCollection('Run configuration changed. Use a new RUN_ID.')
    with db: db.execute("INSERT OR IGNORE INTO metadata VALUES('settings',?)",(settings,))
    try:
        for area in areas:
            crawl_area(db,client,area,1 if INSPECT_ONLY else MAX_PAGES_PER_AREA,FETCH_DETAILS)
            export_store(db,OUTPUT_DIR,RUN_ID)
    finally:
        # Recoverable partial exports on errors and Ctrl-C; checkpoint remains authoritative.
        try: export_store(db,OUTPUT_DIR,RUN_ID)
        finally: db.close()

if RUN_NETWORK:
    run_collection()
else:
    print('Collection disabled. Review configuration and validation notes before enabling.')


Collection disabled. Review configuration and validation notes before enabling.


## 7. Export guide and validation receipt

Files use only `ireland_...` names: checkpoint SQLite, all/sale/rent CSV and JSONL, memberships JSONL, area catalogue JSON, and coverage JSON. CSV is UTF-8 with BOM; JSONL retains numeric/null types. Advertised price ranges/POA remain in `price_text` with null numeric price. Rent periods remain explicit; no monthly conversion is guessed. `region` is the site's region and may be a Dublin postal district. `search_area` belongs to the membership file and is not a certified county/city assignment.

**Checks performed by the author:** all code cells compiled and executed top-to-bottom with network disabled; actual HTML parser checks on the five inspected pages and homepage area discovery; sale/rent IDs, pagination, nonzero search coordinates and detail Eircodes verified; synthetic regression checks for missing state, wrong-page responses, empty results, coordinate placeholders, price ranges, overlap deduplication and checkpoint resume. No full crawl or proxy-connected run was performed. Saved notebook outputs contain only offline execution messages, not scraped datasets. Visual rendering in a Jupyter viewer was not inspected; open the notebook in Jupyter to verify its presentation.

To repeat validation locally: open in Jupyter, Run All with `RUN_NETWORK=False`. For a live smoke test supply your proxy environment variables, review current terms/robots, use a new run ID and enable `RUN_NETWORK` with `INSPECT_ONLY=True`. Review the two-channel output and coverage JSON. If the resolver schema changes, update the adapters against newly inspected HTML; do not add guessed selectors or treat errors as zero listings. Keep HTML fixtures locally if you need reproducible tests; downloaded HTML may contain site-supplied contact fields even though these are excluded from exported records.

BigQuery upload was added 14 November 2026; live cloud validation requires your Colab authentication.


## 8. Upload this run to BigQuery — no further scraping

Destination: **`signalsbi-new.eire_listings_datas.sale_listings_raw`** and
**`signalsbi-new.eire_listings_datas.rental_listings_raw`**. New dataset location: **EU**, matching the reference's EU convention; an existing dataset's actual location is used.

Run this final cell after collection. To add this to your existing Colab notebook, copy this entire code cell into a new final cell; keep your current `RUN_ID` and `OUTPUT_DIR`. No need to repeat collection. The cell reads the SQLite checkpoint, including data committed before an interrupted crawl, rather than relying on potentially stale CSVs.

The uploader creates the dataset/tables if absent and preserves earlier runs. A MERGE keys on `(run_id, channel, listing_id)`: sequential upload retries for the same run do not add duplicate observations; a new run ID preserves a new snapshot. Each channel is committed separately. If the rental upload fails after sales succeeds, rerun only this cell. Do not run simultaneous uploads for the same run. Query jobs and storage may incur Google Cloud charges; uploads do not contact the proxy provider or property site.

`search_areas` preserves all discovered area memberships as a repeated string field. `coverage_complete` is false for inspection/capped/incomplete collection; it does not imply that missing listings are withdrawn. Missing coordinates remain null. Existing incompatible destination schemas raise an error rather than overwrite tables.

Colab will request Google authentication. Your account needs permission to create BigQuery jobs, create the dataset if missing, and create/update tables. Outside Colab, supply Google Application Default Credentials. The BigQuery library is installed if absent. **Local preparation and notebook execution were checked; an authenticated BigQuery upload was not executed here.** Run this cell in Colab to validate the live upload and verify the printed row counts. Notebook presentation in a Jupyter viewer remains unverified.

References: [JSON load jobs](https://cloud.google.com/bigquery/docs/loading-data-cloud-storage-json), [MERGE](https://cloud.google.com/bigquery/docs/reference/standard-sql/dml-syntax#merge_statement).


In [ ]:
# Run this cell alone to upload/retry saved data. No scrape is triggered here.
import json, sqlite3, io, sys, subprocess, uuid
from pathlib import Path
from datetime import datetime, timedelta, timezone

BQ_PROJECT = 'signalsbi-new'
BQ_DATASET = 'eire_listings_datas'
BQ_NEW_DATASET_LOCATION = 'EU'

def prepare_saved_run(folder, run_id):
    checkpoint = Path(folder) / f'ireland_{run_id}_checkpoint.sqlite'
    if not checkpoint.is_file():
        raise FileNotFoundError(f'No checkpoint found at {checkpoint}. Check OUTPUT_DIR and RUN_ID; do not rescrape.')
    db = sqlite3.connect(checkpoint.resolve().as_uri()+'?mode=ro', uri=True)
    try:
        settings_row = db.execute("SELECT value FROM metadata WHERE key='settings'").fetchone()
        settings = json.loads(settings_row[0]) if settings_row else {}
        states = {(c,s):status for c,s,status in db.execute('SELECT channel,slug,status FROM progress')}
        memberships = {}
        for channel,slug,ident in db.execute('SELECT channel,slug,id FROM memberships'):
            memberships.setdefault((channel,ident), []).append(slug)
        grouped = {'sale':[], 'rent':[]}
        complete = {}
        for channel in grouped:
            selected = [a['slug'] for a in settings.get('areas',[]) if a['channel']==channel]
            complete[channel] = bool(selected) and all(states.get((channel,s))=='complete' for s in selected)
        for channel,ident,payload in db.execute('SELECT channel,id,payload FROM listings ORDER BY channel,id'):
            row = json.loads(payload)
            if channel not in grouped or row['channel']!=channel or row['listing_id']!=ident:
                raise ValueError('Checkpoint listing identity/channel mismatch')
            row.update(run_id=run_id, search_areas=sorted(memberships.get((channel,ident),[])),
                       coverage_complete=complete[channel])
            grouped[channel].append(row)
        if not any(grouped.values()): raise ValueError('Checkpoint has no saved listings to upload.')
        return grouped
    finally:
        db.close()

def upload_saved_run_to_bigquery(folder, run_id):
    grouped = prepare_saved_run(folder, run_id)
    try:
        from google.cloud import bigquery
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','google-cloud-bigquery'])
        from google.cloud import bigquery
    try:
        from google.colab import auth
    except ImportError:
        pass
    else:
        auth.authenticate_user()
    client = bigquery.Client(project=BQ_PROJECT)
    dataset_id = f'{BQ_PROJECT}.{BQ_DATASET}'
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = BQ_NEW_DATASET_LOCATION
    client.create_dataset(dataset, exists_ok=True)
    location = client.get_dataset(dataset_id).location

    # Explicit schema prevents all-null coordinate columns becoming strings.
    float_fields = {'asking_price_eur','floor_area_m2','floor_area_ft2','latitude','longitude'}
    int_fields = {'bedrooms','bathrooms'}
    bool_fields = {'is_sale_agreed','is_private_landlord','is_mapped_accurately','coverage_complete'}
    timestamp_fields = {'collected_at'}
    columns = ['listing_id','channel','source_url','address','eircode','locality','region',
        'price_text','asking_price_eur','rent_period','bedrooms','bathrooms','floor_area_m2',
        'floor_area_ft2','property_type','ber_rating','listing_status','is_sale_agreed',
        'is_private_landlord','agent_name','created_on','activated_on','refreshed_on',
        'latitude','longitude','coordinate_source','is_mapped_accurately','detail_status',
        'collected_at','run_id','search_areas','coverage_complete']
    schema = []
    for name in columns:
        dtype = ('FLOAT' if name in float_fields else 'INTEGER' if name in int_fields else
                 'BOOLEAN' if name in bool_fields else 'TIMESTAMP' if name in timestamp_fields else 'STRING')
        schema.append(bigquery.SchemaField(name,dtype,mode='REPEATED' if name=='search_areas' else 'NULLABLE'))
    for channel,table_name in [('sale','sale_listings_raw'),('rent','rental_listings_raw')]:
        rows = grouped[channel]
        destination = f'{dataset_id}.{table_name}'
        client.create_table(bigquery.Table(destination,schema=schema),exists_ok=True)
        if not rows:
            print(destination+': no saved rows for this run; table left unchanged.')
            continue
        stage_id = f'{dataset_id}._ireland_upload_{uuid.uuid4().hex}'
        stage = bigquery.Table(stage_id,schema=schema)
        stage.expires = datetime.now(timezone.utc)+timedelta(days=1)
        client.create_table(stage)
        # Fresh staging table per attempt. Raw destination tables are never truncated.
        try:
            stream = io.BytesIO((''.join(json.dumps(row,ensure_ascii=False)+'\n' for row in rows)).encode('utf-8'))
            config = bigquery.LoadJobConfig(schema=schema,
                source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
                write_disposition=bigquery.WriteDisposition.WRITE_EMPTY)
            client.load_table_from_file(stream,stage_id,job_config=config,location=location).result()
            names = ', '.join('`'+c+'`' for c in columns)
            values = ', '.join('S.`'+c+'`' for c in columns)
            updates = ', '.join('T.`'+c+'` = S.`'+c+'`' for c in columns
                                if c not in {'run_id','channel','listing_id'})
            query = f"""MERGE `{destination}` T USING `{stage_id}` S
                ON T.run_id=S.run_id AND T.channel=S.channel AND T.listing_id=S.listing_id
                WHEN MATCHED THEN UPDATE SET {updates}
                WHEN NOT MATCHED THEN INSERT ({names}) VALUES ({values})"""
            client.query(query,location=location).result()
            verify = client.query(f'SELECT COUNT(*) AS n FROM `{destination}` WHERE run_id=@run_id AND channel=@channel',
                location=location,job_config=bigquery.QueryJobConfig(query_parameters=[
                    bigquery.ScalarQueryParameter('run_id','STRING',run_id),
                    bigquery.ScalarQueryParameter('channel','STRING',channel)])).result()
            count = next(iter(verify)).n
            if count != len(rows): raise RuntimeError(f'Uploaded row count differs: expected {len(rows)}, found {count}')
            print(f'{destination}: verified {count} rows for run {run_id}; coverage_complete={rows[0]["coverage_complete"]}')
        finally:
            try: client.delete_table(stage_id,not_found_ok=True)
            except Exception: print('Temporary upload table will expire automatically within one day.')
    print('BigQuery upload complete. Local checkpoint and exports retained.')

if RUN_NETWORK:
    upload_saved_run_to_bigquery(OUTPUT_DIR, RUN_ID)
else:
    print('BigQuery upload not run in offline mode. To upload existing data only, call:')
    print('upload_saved_run_to_bigquery(OUTPUT_DIR, RUN_ID)')


BigQuery upload not run in offline mode. To upload existing data only, call:
upload_saved_run_to_bigquery(OUTPUT_DIR, RUN_ID)
